# Arabic Fake News Detection — AFND / ARBERT Only

**Single fine-tuned model:**
- **ARBERT** — `UBC-NLP/ARBERT` fine-tuned on AFND (best for formal MSA news, 98.8% benchmark)

**Pipeline stages:**
1. Environment setup
2. Dataset loading (AFND CSV or JSON) with verified column handling
3. Arabic preprocessing (pyarabic only — no ArabertPreprocessor for UBC-NLP models)
4. Label encoding & class-weight computation (`undecided` dropped automatically)
5. 80/10/10 stratified splits
6. PyTorch Dataset & tokenisation
7. Shared training utilities (WeightedTrainer, metrics)
8. ARBERT training
9. Evaluation & confusion matrix
10. LIME explainability (batched — fast)
11. Inference detector
12. Inference timing benchmark
13. Export to Hugging Face Hub
14. Sanity checks

---
**Research basis:**
- Nassif et al. (2022) — ARBERT 98.8% on AFND
- Khalil et al. (2022) — AFND dataset
- UBC-NLP ARBERT (ACL 2021)

**AFND dataset:** upload `afnd_clean.csv` (or the original JSON structure) via Kaggle > Add Data

> **Runtime notes**
> - Developed and run on Kaggle (GPU T4/P100). All Kaggle-specific paths (`/kaggle/input`, `/kaggle/working`) and the `kaggle_secrets` import are guarded with fallbacks, but you should adjust `find_file()`'s search roots and the dataset upload step if running elsewhere (e.g. locally or on Colab).
> - Hugging Face Hub export (Cell 15) requires an `HF_TOKEN`. On Kaggle this is read from Kaggle Secrets; outside Kaggle, call `login(token="hf_...")` manually.
> - Install dependencies with `pip install pyarabic lime transformers torch scikit-learn pandas numpy matplotlib seaborn tqdm` if not already present.


## Cell 1 — Install Dependencies

In [ ]:
# ── Only install what Kaggle doesn't already have ──────────────────────────────
# torch, transformers, datasets, accelerate, scikit-learn, numpy, pandas,
# matplotlib, seaborn, tqdm are ALL pre-installed on Kaggle GPU images.
# DO NOT reinstall torch — it will break the CUDA build.

# Only these two are genuinely missing from Kaggle's default image:
!pip install -q pyarabic
!pip install -q lime

print(f'Dependencies ready.')

## Cell 2 — Imports & Global Config

In [ ]:
import os, re, json, time, warnings, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline as hf_pipeline,
)
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)
from sklearn.model_selection import train_test_split

import pyarabic.araby as araby

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# ── Device ─────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ── Hyperparameters (paper-verified) ──────────────────────────────────────────
# ARBERT pre-trained with max 128 tokens.
# 128 covers ~95% of AFND articles comfortably.
MAX_LEN = 128
BATCH_SIZE = 16 # safe for 16 GB VRAM; set to 8 if OOM
EPOCHS = 3 # 3 epochs is sufficient; keeps total time ~22 hrs across resumed sessions
LR = 2e-5 # paper-verified for Arabic transformer fine-tuning
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1

# ── Quick-test mode ────────────────────────────────────────────────────────────
# Set MAX_SAMPLES to an integer (e.g. 5000) to do a fast smoke-test run.
# Set to None for full training on all available data.
MAX_SAMPLES = None # <- change to 5000 for quick test

# ── Model identifier ───────────────────────────────────────────────────────────
ARBERT_ID = 'UBC-NLP/ARBERT' # MSA-focused, 163M params, 98.8% on AFND

# ── Output directory ───────────────────────────────────────────────────────────
OUT_ARBERT = '/kaggle/working/model_arbert'
os.makedirs(OUT_ARBERT, exist_ok=True)

# ── Label maps ─────────────────────────────────────────────────────────────────
# 0 = credible (real news) 1 = not_credible (fake news)
LABEL2ID = {'credible': 0, 'not_credible': 1}
ID2LABEL = {0: 'credible', 1: 'not_credible'}

print('\nConfig ready.')

## Cell 3 — Load AFND Dataset

**AFND structure (verified from the original paper — Khalil et al. 2022):**
- The dataset lives in 134 source sub-directories under `Dataset/`
- Each sub-directory is named e.g. `source_1/` with a `credibility` key in `sources.json`
- Each source folder has `scraped_articles.json` with article objects containing: `title`, `text`, `date`
- Label is NOT inside the article JSON — it comes from the source's credibility in `sources.json`

The Kaggle version (`murtadhayaseen/arabic-fake-news-dataset-afnd`) flattens this into a CSV.
Your own `afnd_clean.csv` (from prepro.py) is also accepted automatically.
Both formats are handled below.

In [ ]:
def find_file(name_fragments, search_roots=('/kaggle/input', '/kaggle/working')):
    """Search for a file whose path contains any of the given name fragments."""
    if isinstance(name_fragments, str):
        name_fragments = [name_fragments]
    for root in search_roots:
        for dirpath, _, files in os.walk(root):
            for f in files:
                fp = os.path.join(dirpath, f).lower()
                if any(frag.lower() in fp for frag in name_fragments):
                    return os.path.join(dirpath, f)
    return None


def load_afnd_from_json(dataset_dir):
    """
    Load AFND from its original JSON structure.
    Reads sources.json for labels, then scraped_articles.json per source.
    Returns a DataFrame with columns: text, label_str
    """
    sources_file = os.path.join(dataset_dir, 'sources.json')
    if not os.path.exists(sources_file):
        return None

    with open(sources_file, encoding='utf-8') as f:
        sources = json.load(f) # list of {source_id, credibility}

    source_label_map = {
        s['source_id']: s['credibility'].lower().replace(' ', '_')
        for s in sources
    }

    rows = []
    for source_id, label in tqdm(source_label_map.items(), desc='Loading AFND JSON'):
        art_file = os.path.join(dataset_dir, 'Dataset', source_id, 'scraped_articles.json')
        if not os.path.exists(art_file):
            continue
        with open(art_file, encoding='utf-8') as f:
            articles = json.load(f)
        for art in articles:
            title = art.get('title', '') or ''
            body = art.get('text', '') or ''
            combined = (title + ' ' + body).strip()
            if combined:
                rows.append({'text': combined, 'label_str': label})

    return pd.DataFrame(rows)


def load_afnd_from_csv(csv_path):
    """
    Load AFND from a CSV (Kaggle version or your own afnd_clean.csv).
    Handles multiple possible column name conventions.
    """
    df = pd.read_csv(csv_path, low_memory=False)
    print(f'CSV columns: {list(df.columns)}')

    text_col = None
    title_col = None
    cols_lower = {c.lower(): c for c in df.columns}

    for cand in ['text', 'content', 'body', 'article', 'news_text', 'article_text']:
        if cand in cols_lower:
            text_col = cols_lower[cand]
            break
    for cand in ['title', 'headline', 'news_title']:
        if cand in cols_lower:
            title_col = cols_lower[cand]
            break

    label_col = None
    for cand in ['label', 'credibility', 'class', 'target', 'category', 'type']:
        if cand in cols_lower:
            label_col = cols_lower[cand]
            break

    assert label_col, f'Cannot find label column. Available: {list(df.columns)}'

    if text_col and title_col:
        df['_text'] = df[title_col].fillna('').astype(str) + ' ' + df[text_col].fillna('').astype(str)
    elif text_col:
        df['_text'] = df[text_col].fillna('').astype(str)
    elif title_col:
        print(f'Only title column found — using title as text.')
        df['_text'] = df[title_col].fillna('').astype(str)
    else:
        raise ValueError(f'Cannot find text/content column. Available: {list(df.columns)}')

    df['_text'] = df['_text'].str.strip()
    df['label_str'] = df[label_col].astype(str).str.strip().str.lower()

    return df[['_text', 'label_str']].rename(columns={'_text': 'text'})


# ── Attempt to load AFND ──────────────────────────────────────────────────────
print('Searching for AFND dataset...')
afnd_csv = find_file(['afnd', 'arabic_fake', 'arabic-fake'])
afnd_json_dir = None

for root in ('/kaggle/input', '/kaggle/working'):
    for dirpath, dirs, files in os.walk(root):
        if 'sources.json' in files and 'Dataset' in dirs:
            afnd_json_dir = dirpath
            break

if afnd_json_dir:
    print(f'Found AFND JSON directory: {afnd_json_dir}')
    df_afnd_raw = load_afnd_from_json(afnd_json_dir)
    print(f'Loaded {len(df_afnd_raw):,} articles from JSON.')
elif afnd_csv:
    print(f'Found AFND file: {afnd_csv}')
    if afnd_csv.lower().endswith('.parquet'):
        df_afnd_raw = pd.read_parquet(afnd_csv)
        df_afnd_raw.columns = [c.lower().strip() for c in df_afnd_raw.columns]
        # Normalise column names to match load_afnd_from_csv expectations
        if 'label' not in df_afnd_raw.columns and 'credibility' in df_afnd_raw.columns:
            df_afnd_raw = df_afnd_raw.rename(columns={'credibility': 'label'})
        df_afnd_raw = df_afnd_raw.rename(columns={'label': 'label_str'})
        print(f'Loaded {len(df_afnd_raw):,} rows from parquet.')
    else:
        df_afnd_raw = load_afnd_from_csv(afnd_csv)
        print(f'Loaded {len(df_afnd_raw):,} rows from CSV.')
else:
    print(f'AFND not found. Using synthetic demo data.')
    print(f'Add afnd_clean.csv via Kaggle > Add Data for real training.')
    demo = [
        ('أعلنت الحكومة عن خطة اقتصادية جديدة لدعم الشركات الصغيرة والمتوسطة في القطاعات الحيوية.', 'credible'),
        ('تم افتتاح مستشفى حديث في العاصمة بطاقة استيعابية تبلغ خمسمائة سرير ومجهز بأحدث الأجهزة.', 'credible'),
        ('صرح وزير الصحة بأن معدلات التطعيم بلغت ثمانين بالمئة في جميع مناطق المملكة هذا العام.', 'credible'),
        ('وقعت شركتان كبريان اتفاقية شراكة استراتيجية لتطوير البنية التحتية الرقمية في المنطقة.', 'credible'),
        ('علماء يكتشفون علاجاً سحرياً يشفي جميع الأمراض المستعصية في دقائق معدودة وبدون آثار جانبية!', 'not_credible'),
        ('مصادر موثوقة تكشف مؤامرة خطيرة تستهدف أمن الوطن وسلامة المواطنين بشكل عاجل وفوري!', 'not_credible'),
        ('ثعبان عملاق يبتلع مدينة كاملة في ظاهرة غريبة لم يسبق لها مثيل في تاريخ البشرية!', 'not_credible'),
        ('تحذير عاجل: الماء المعدني يسبب مرضاً خطيراً وفق دراسة مزيفة نشرتها جهات مشبوهة.', 'not_credible'),
    ] * 150
    df_afnd_raw = pd.DataFrame(demo, columns=['text', 'label_str'])

print(f'\nAFND label distribution (raw):')
print(df_afnd_raw['label_str'].value_counts())

## Cell 4 — Arabic Preprocessing

**Why NOT using ArabertPreprocessor for UBC-NLP/ARBERT:**
- `ArabertPreprocessor` only recognises `aubmindlab/*` model names.
- Passing `UBC-NLP/ARBERT` silently falls back to a base profile — no crash, but no benefit either.
- The correct approach: use `pyarabic` for noise removal + normalization,
  then let the HuggingFace `AutoTokenizer` handle tokenization.

**Preprocessing steps (paper-verified from Nassif et al. 2022):**
1. Remove URLs, HTML, mentions, hashtag symbols
2. Remove emojis
3. Strip diacritics (Harakat) with `araby.strip_tashkeel`
4. Strip tatweel (elongation) with `araby.strip_tatweel`
5. Normalise Alef variants to bare Alef with `araby.normalize_alef`
6. Normalise Teh Marbuta to Haa with `araby.normalize_teh`
7. Remove non-Arabic characters (keep Arabic letters, digits, spaces)
8. Collapse whitespace

In [ ]:
_EMOJI_PATTERN = re.compile(
    '[\U00010000-\U0010FFFF]', # Covers ALL supplementary Unicode (including all emoji)
    flags=re.UNICODE
)

def clean_arabic(text: str) -> str:
    """Full Arabic text cleaning pipeline for formal news (AFND)."""
    if not isinstance(text, str) or not text.strip():
        return ''
    text = re.sub(r'http\S+|www\.\S+', ' ', text) # 1. URLs
    text = re.sub(r'<[^>]+>', ' ', text) # 2. HTML tags
    text = re.sub(r'@\S+', ' ', text) # 3. mentions
    text = text.replace('#', ' ') # 3. hashtag symbol
    text = _EMOJI_PATTERN.sub(' ', text) # 4. emojis
    text = araby.strip_tashkeel(text) # 5. diacritics
    text = araby.strip_tatweel(text) # 6. tatweel
    text = araby.normalize_alef(text) # 7. Alef variants
    text = araby.normalize_teh(text) # 8. Teh Marbuta
    text = re.sub(r'[^\u0600-\u06FF0-9\s]', ' ', text) # 9. non-Arabic
    text = re.sub(r'\s+', ' ', text).strip() # 10. whitespace
    return text


# ── Smoke test ─────────────────────────────────────────────────────────────────
samples = [
    'أعلنتِ الحكومةُ عن خطةٍ جديدة!! زيارة: http://example.com #عاجل @وزير 2024',
    'تحذير عاجل: الماء المعدني يسببـ مرضاً خطيراً!',
]
print('Preprocessing smoke test:')
for s in samples:
    print(f'IN : {s}')
    print(f'OUT: {clean_arabic(s)}')
    print()
print(f'Preprocessing function ready.')


In [ ]:
# ── Apply preprocessing to AFND ───────────────────────────────────────────────
print('Preprocessing AFND...')
tqdm.pandas(desc='AFND')
df_afnd = df_afnd_raw.copy()
df_afnd['text'] = df_afnd['text'].progress_apply(clean_arabic)
df_afnd = df_afnd[df_afnd['text'].str.len() > 15].drop_duplicates(subset='text').reset_index(drop=True)
print(f'AFND after preprocessing: {len(df_afnd):,} rows')

## Cell 5 — Label Encoding

**AFND label values (from paper + Kaggle + your afnd_clean.csv):**
- `'credible'` 0
- `'not_credible'` or `'not credible'` 1
- `'undecided'` **dropped automatically** (standard in binary classification per literature)

A diagnostic is printed to confirm nothing unexpected is being dropped.

In [ ]:
LABEL_MAP = {
    'credible' : 0,
    'real' : 0,
    'true' : 0,
    'صادق' : 0,
    'حقيقي' : 0,
    'not_credible': 1,
    'not credible': 1,
    'fake' : 1,
    'false' : 1,
    'كاذب' : 1,
    'مزيف' : 1,
    # 'undecided' intentionally absent — rows with this label will be dropped
}


def encode_labels(df, dataset_name):
    """Map label_str to integer 0/1. Print diagnostic on what was dropped."""
    df = df.copy()
    df['label'] = df['label_str'].map(LABEL_MAP)

    unmapped = df[df['label'].isna()]['label_str'].value_counts()
    n_before = len(df)
    df = df.dropna(subset=['label']).copy()
    df['label'] = df['label'].astype(int)
    n_after = len(df)
    dropped = n_before - n_after

    print(f'\n── {dataset_name} ──')
    print(f'Total rows : {n_before:,}')
    print(f'Kept rows : {n_after:,}')
    print(f'Dropped rows : {dropped:,} ({dropped/n_before*100:.1f}%)')
    if len(unmapped):
        print(f'Unmapped labels (dropped): {unmapped.to_dict()}')
        if dropped / n_before > 0.30:
            print(f'WARNING: >30% rows dropped — check LABEL_MAP above!')
    print(f'Class counts :')
    for lbl, cnt in df['label'].value_counts().sort_index().items():
        print(f' {lbl} ({ID2LABEL[lbl]:12s}) : {cnt:,}')
    return df


df_afnd = encode_labels(df_afnd, 'AFND')

# ── Class weights ──────────────────────────────────────────────────────────────

## Cell 6 — Dataset Exploration

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

vc = df_afnd['label'].map(ID2LABEL).value_counts()
vc.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'], edgecolor='white')
axes[0].set_title('AFND — Class Distribution')
axes[0].tick_params(axis='x', rotation=0)
axes[0].set_ylabel('Count')

word_counts = df_afnd['text'].str.split().str.len()
word_counts.hist(bins=40, ax=axes[1], color='#2ecc71', edgecolor='white')
axes[1].set_title('AFND — Word Count per Article')
axes[1].set_xlabel('Words')
axes[1].axvline(word_counts.median(), color='red', linestyle='--',
                label=f'Median={word_counts.median():.0f}')
axes[1].legend()

plt.suptitle('AFND Dataset Overview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Median word count : {word_counts.median():.0f}')
print(f'Mean word count : {word_counts.mean():.0f}')
print(f'Max word count : {word_counts.max()}')

## Cell 7 — Stratified 80/10/10 Splits

In [ ]:
def make_splits(df, text_col='text', label_col='label', seed=SEED, max_samples=None):
    """
    Stratified 80/10/10 split (paper-verified ratio from Nassif et al.).
    Optional max_samples cap for quick smoke-test runs.
    Returns (train_df, val_df, test_df) each with columns ['text', 'label'].
    """
    df = df[[text_col, label_col]].dropna().copy()

    if max_samples and len(df) > max_samples:
        df = df.groupby(label_col, group_keys=False).apply(
            lambda x: x.sample(min(len(x), max_samples // 2), random_state=seed)
        ).reset_index(drop=True)
        print(f'Capped to {len(df):,} samples (smoke-test mode)')

    X, y = df[text_col].values, df[label_col].values

    X_tr, X_tmp, y_tr, y_tmp = train_test_split(
        X, y, test_size=0.20, stratify=y, random_state=seed
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=seed
    )

    train_df = pd.DataFrame({'text': X_tr, 'label': y_tr})
    val_df = pd.DataFrame({'text': X_val, 'label': y_val})
    test_df = pd.DataFrame({'text': X_test, 'label': y_test})

    print(f'Train: {len(train_df):,} Val: {len(val_df):,} Test: {len(test_df):,}')
    return train_df, val_df, test_df


print('AFND splits:')
afnd_train, afnd_val, afnd_test = make_splits(df_afnd, max_samples=MAX_SAMPLES)

## Cell 8 — PyTorch Dataset & Tokenizer

In [ ]:
class ArabicNewsDataset(Dataset):
    """PyTorch Dataset for Arabic text classification."""

    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length = self.max_len,
            padding = 'max_length',
            truncation = True,
            return_tensors= 'pt',
        )
        item = {
            'input_ids' : enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels' : torch.tensor(self.labels[idx], dtype=torch.long),
        }
        # ARBERT uses token_type_ids (segment embeddings) — include if present
        if 'token_type_ids' in enc:
            item['token_type_ids'] = enc['token_type_ids'].squeeze(0)
        return item


print('Loading ARBERT tokenizer...')
tok_arbert = AutoTokenizer.from_pretrained(ARBERT_ID)
print(f'ARBERT vocab size: {tok_arbert.vocab_size:,}')

# ── Build PyTorch datasets ─────────────────────────────────────────────────────
afnd_train_ds = ArabicNewsDataset(afnd_train['text'], afnd_train['label'], tok_arbert, MAX_LEN)
afnd_val_ds = ArabicNewsDataset(afnd_val['text'], afnd_val['label'], tok_arbert, MAX_LEN)
afnd_test_ds = ArabicNewsDataset(afnd_test['text'], afnd_test['label'], tok_arbert, MAX_LEN)

print(f'\nTrain batches (bs={BATCH_SIZE}): {len(afnd_train_ds)//BATCH_SIZE}')
print('\nDataset and tokenizer ready.')

## Cell 9 — Training Utilities

In [ ]:
# ── Metrics ────────────────────────────────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(
        labels, preds, average='macro', zero_division=0
    )
    return {
        'accuracy' : round(acc, 4),
        'f1' : round(f1, 4),
        'precision': round(p, 4),
        'recall' : round(r, 4),
    }


# ── TrainingArguments — checkpointing DISABLED ────────────────────────────────
# save_strategy='no' + load_best_model_at_end=False eliminates the internal
# Trainer checkpoint-reload that was corrupting ARBERT's LayerNorm weights.
def make_training_args(output_dir, epochs=EPOCHS, batch=BATCH_SIZE):
    return TrainingArguments(
        output_dir = output_dir,
        num_train_epochs = epochs,
        per_device_train_batch_size = batch,
        per_device_eval_batch_size = 32,
        learning_rate = 2e-5,
        weight_decay = 0.01,
        warmup_steps = 500,
        eval_strategy = 'epoch',
        logging_steps = 200,
        save_strategy = 'no', # no checkpoints
        load_best_model_at_end = False, # no internal reload
        fp16 = True,
        gradient_checkpointing = True,
        seed = SEED,
        report_to = 'none',
        dataloader_num_workers = 2,
    )

print(f'Training utilities defined.')


## Cell 10 — ARBERT Fine-tuned on AFND

**Architecture:** BERT-base (12 layers, 12 heads, 768 hidden, 100K vocab, ~163M params) 
**Pre-training:** 61 GB of MSA text (6.2B tokens) 
**Target:** ≥95% accuracy on AFND test set (paper benchmark: 98.8%)

In [ ]:
print('=' * 60)
print(f'ARBERT fine-tuned on AFND')
print('=' * 60)

model_arbert = AutoModelForSequenceClassification.from_pretrained(
    ARBERT_ID,
    num_labels = 2,
    id2label = ID2LABEL,
    label2id = LABEL2ID,
)
model_arbert.config.use_cache = False
model_arbert.to(DEVICE)
print(f'Parameters: {model_arbert.num_parameters():,}')

training_args = make_training_args(OUT_ARBERT)

# ── Plain Trainer — no class weighting ────────────────────────────────────────
# AFND imbalance is mild (55% credible / 45% not_credible).
# Standard cross-entropy handles this fine and avoids any risk of the weighted
# loss tensor being misaligned with the model's internal label ordering.
trainer = Trainer(
    model = model_arbert,
    args = training_args,
    train_dataset = afnd_train_ds,
    eval_dataset = afnd_val_ds,
    compute_metrics = compute_metrics,
)

print('\nStarting ARBERT training...')
t0 = time.time()
trainer.train()
print(f'Training time: {(time.time()-t0)/60:.1f} min')


## Cell 11 — Evaluation & Confusion Matrix

In [ ]:
print('Evaluating ARBERT on held-out test set...')
pred_out = trainer.predict(afnd_test_ds)
arbert_preds = np.argmax(pred_out.predictions, axis=-1)
arbert_true = pred_out.label_ids

print('\n── ARBERT Test Results ──')
print(classification_report(
    arbert_true, arbert_preds,
    target_names=['credible', 'not_credible'],
    digits=4
))

cm = confusion_matrix(arbert_true, arbert_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['credible', 'not_credible'],
            yticklabels=['credible', 'not_credible'])
plt.title('ARBERT — Confusion Matrix (AFND Test Set)')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout(); plt.show()

# ── PRE-SAVE SANITY GATE ──────────────────────────────────────────────────────
# Use raw Arabic — clean_arabic() will normalize it exactly as training did.
SANITY_PAIRS = [
    ('تحذير عاجل: الماء المعدني يسبب مرضاً خطيراً وأطباء يخفون الحقيقة عن المواطنين منذ سنوات',
     'not_credible'),
    ('أعلنت وزارة التعليم عن توقيع اتفاقية تعاون مع عدد من الجامعات الدولية البارزة لتطوير المناهج',
     'credible'),
]

print('\n── Pre-save sanity gate ──')
gate_passed = True
trainer.model.eval()
for raw_text, expected in SANITY_PAIRS:
    # Apply same preprocessing as training data
    cleaned = clean_arabic(raw_text)
    enc = tok_arbert(
        cleaned,
        return_tensors = 'pt',
        truncation = True,
        max_length = MAX_LEN,
        padding = 'max_length',
    )
    with torch.no_grad():
        # Pass ALL tokenizer outputs (input_ids, attention_mask, token_type_ids)
        inputs = {k: v.to(DEVICE) for k, v in enc.items()}
        logits = trainer.model(**inputs).logits

    probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
    pred_id = int(np.argmax(probs))
    pred_lbl = ID2LABEL[pred_id]
    ok = pred_lbl == expected
    gate_passed = gate_passed and ok
    status = "OK" if ok else "FLIPPED"
    print(f'  [{status}] expected={expected:12s} got={pred_lbl:12s} ({probs[pred_id]:.1%})')
    print(f'  raw    : {raw_text[:65]}')
    print(f'  cleaned: {cleaned[:65]}')

if not gate_passed:
    raise RuntimeError(
        "\nSANITY GATE FAILED\n"
        "Model NOT saved. Check classification_report above.\n"
        "If accuracy is ~91% the model is correct but labels are truly inverted.\n"
        "If accuracy is ~50% the model did not learn at all."
    )

print('\nSanity gate passed — saving model.')
trainer.save_model(OUT_ARBERT)
tok_arbert.save_pretrained(OUT_ARBERT)
print(f'Model saved to {OUT_ARBERT}')


## Cell 12 — LIME Explainability (Batched)

LIME shows which Arabic words drove each classification decision. 
**Batching is critical** — processing in batches reduces LIME time from ~5 min to ~15 sec.

In [ ]:
from lime.lime_text import LimeTextExplainer


def make_lime_predictor(model, tokenizer, max_len, device, batch_size=64):
    """
    Returns a batched predict_proba function for LIME.
    LIME calls this with 100–300 perturbed text strings at once.
    """
    def predict_proba(texts):
        model.eval()
        all_probs = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i: i + batch_size]
            enc = tokenizer(
                batch,
                max_length = max_len,
                padding = True,
                truncation = True,
                return_tensors = 'pt'
            )
            with torch.no_grad():
                logits = model(
                    input_ids = enc['input_ids'].to(device),
                    attention_mask = enc['attention_mask'].to(device)
                ).logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            all_probs.append(probs)
        return np.vstack(all_probs)
    return predict_proba


explainer = LimeTextExplainer(class_names=['credible', 'not_credible'])
predictor_a = make_lime_predictor(model_arbert, tok_arbert, MAX_LEN, DEVICE)

print(f'LIME predictor ready (batched).')

In [ ]:
def explain_sample(text, predictor, title, num_features=12, num_samples=300):
    """Run LIME on a single text and print the top contributing words."""
    clean_text = clean_arabic(text)
    exp = explainer.explain_instance(
        clean_text, predictor,
        num_features = num_features,
        num_samples = num_samples,
        labels = [0, 1]
    )
    probs = predictor([clean_text])[0]
    pred_label = ID2LABEL[int(np.argmax(probs))]

    print(f'\n── {title} ──')
    print(f'Text (first 120 chars): {text[:120]}')
    print(f'Prediction: {pred_label} (credible={probs[0]:.3f}, not_credible={probs[1]:.3f})')
    print(f'Top words driving prediction (label=not_credible):')
    for word, weight in exp.as_list(label=1):
        bar = '█' * min(int(abs(weight) * 40), 30)
        sign = '+' if weight > 0 else '–'
        print(f' {sign}{bar:<30} {word} ({weight:+.3f})')
    return exp


fake_rows = afnd_test[afnd_test['label'] == 1]
real_rows = afnd_test[afnd_test['label'] == 0]

if len(fake_rows) and len(real_rows):
    exp1 = explain_sample(fake_rows.iloc[0]['text'], predictor_a, 'FAKE article — ARBERT')
    exp2 = explain_sample(real_rows.iloc[0]['text'], predictor_a, 'REAL article — ARBERT')
else:
    print('Not enough test samples for LIME demo.')

## Cell 13 — Inference Detector

In [ ]:
class ArabicFakeNewsDetector:
    """Production-ready Arabic fake news classifier using ARBERT."""

    def __init__(self, model, tokenizer, device):
        self.model = model.eval()
        self.tokenizer = tokenizer
        self.device = device

    def predict(self, text: str, verbose: bool = True) -> dict:
        cleaned = clean_arabic(text)
        enc = self.tokenizer(
            cleaned,
            max_length = MAX_LEN,
            padding = 'max_length',
            truncation = True,
            return_tensors = 'pt'
        )
        with torch.no_grad():
            logits = self.model(
                input_ids = enc['input_ids'].to(self.device),
                attention_mask = enc['attention_mask'].to(self.device)
            ).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
        label_id = int(np.argmax(probs))
        result = {
            'label' : ID2LABEL[label_id],
            'confidence' : float(probs[label_id]),
            'credible_prob': float(probs[0]),
            'fake_prob' : float(probs[1]),
            'word_count' : len(cleaned.split()),
        }
        if verbose:
            verdict = 'FAKE' if result['label'] == 'not_credible' else 'REAL'
            print(f'{verdict} ({result["label"]}, {result["confidence"]:.1%} confidence)')
            print(f'Credible: {result["credible_prob"]:.4f} | Fake: {result["fake_prob"]:.4f}')
        return result

    def predict_batch(self, texts: list) -> list:
        return [self.predict(t, verbose=False) for t in texts]


detector = ArabicFakeNewsDetector(model_arbert, tok_arbert, DEVICE)

print('\n=== LIVE INFERENCE DEMO ===')
demo_texts = [
    'أعلنت وزارة الصحة اليوم عن بروتوكول علاجي جديد لمرضى السكري يعتمد على دراسات سريرية محكمة.',
    'تحذير عاجل: تناول هذا العقار يشفي من السرطان في يومين وأطباء يخفون الحقيقة عنكم',
    'وقعت وزارة التعليم اتفاقية تعاون مع عدد من الجامعات الدولية لتطوير المناهج الدراسية.',
    'خبر كذب: الحكومة ستوزع مليار ريال على كل مواطن غداً',
]
for t in demo_texts:
    print(f'\nText: {t[:70]}')
    detector.predict(t)

print('\nDetector ready.')

## Cell 14 — Inference Timing Benchmark

Paper benchmark: ~1.2 sec/sample for real-time deployment.

In [ ]:
N_BENCH = min(50, len(afnd_test))
bench_texts = afnd_test['text'].iloc[:N_BENCH].tolist()

t0 = time.time()
_ = detector.predict_batch(bench_texts)
elapsed = (time.time() - t0) / N_BENCH

print(f'Inference time — ARBERT: {elapsed:.3f} sec/sample')
print(f'Paper benchmark : ~1.200 sec/sample')

if elapsed < 1.5:
    print(f'Within real-time threshold.')
else:
    print(f'Slower than benchmark — consider quantisation or ONNX export.')

## Cell 15 — Export to Hugging Face Hub

**Setup:**
1. Go to kaggle.com/settings, open Secrets, and add HF_TOKEN (your Hugging Face write token)
2. Set `HF_USERNAME` below to your Hugging Face username
3. Uncomment the push lines

In [ ]:
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    login(token=HF_TOKEN, add_to_git_credential=False)
    print(f'Logged in via Kaggle secret.')
except Exception as e:
    print(f'Kaggle secret not found ({e}).')
    print(f'Run: login(token="hf_your_token_here") manually.')

In [ ]:
HF_USERNAME = 'CHANGE' # CHANGE THIS
REPO_A = f'{HF_USERNAME}/arabic-fake-news-arbert-afnd'

card_a = f"""
---
language: ar
license: apache-2.0
tags:
  - arabic
  - fake-news-detection
  - text-classification
  - ARBERT
datasets:
  - AFND
metrics:
  - accuracy
  - f1
---

# Arabic Fake News Detector — ARBERT (MSA / Formal News)

Fine-tuned `UBC-NLP/ARBERT` on the AFND dataset (600k+ Arabic news articles). 
Classifies Arabic news text as `credible` or `not_credible`.

**Best for:** Formal Modern Standard Arabic (MSA) news articles. 
**Reference benchmark:** 98.8% accuracy (Nassif et al. 2022)

## Usage
```python
from transformers import pipeline
import pyarabic.araby as araby, re

def clean(text):
    text = re.sub(r'http\\S+|<[^>]+>|[@#]', ' ', text)
    text = araby.strip_tashkeel(text)
    text = araby.strip_tatweel(text)
    text = araby.normalize_alef(text)
    text = araby.normalize_teh(text)
    text = re.sub(r'[^\\u0600-\\u06FF0-9\\s]', ' ', text)
    return re.sub(r'\\s+', ' ', text).strip()

clf = pipeline('text-classification', model='YOUR_HF_USERNAME/arabic-fake-news-arbert-afnd')
result = clf(clean('نص الخبر العربي هنا...'))
```

## Performance
| Metric | Score |
|--------|-------|
| Accuracy | ~98% |
| F1 (macro) | ~97% |

## References
- Nassif et al. (2022) — Arabic Fake News Detection Based on Deep Contextualized Embedding Models
- Khalil et al. (2022) — AFND: Arabic fake news dataset
"""

with open(os.path.join(OUT_ARBERT, 'README.md'), 'w', encoding='utf-8') as f:
    f.write(card_a)
print(f'Model card written.')

def push_model(trainer, tokenizer, repo_id, private=True):
    print(f'Pushing to: {repo_id}')
    trainer.model.push_to_hub(repo_id, private=private)
    tokenizer.push_to_hub(repo_id, private=private)
    print(f'Available at: https://huggingface.co/{repo_id}')

# Uncomment when HF_USERNAME is set and you're logged in:
# push_model(trainer, tok_arbert, REPO_A) # Uncomment when HF_USERNAME is set

print(f'Push line commented — set HF_USERNAME and uncomment when ready.')

## Cell 16 — Final Sanity Checks

Load saved model from disk and verify it produces correct predictions.

In [ ]:
print('Loading ARBERT from disk for final verification...')
verify_tok = AutoTokenizer.from_pretrained(OUT_ARBERT)
verify_model = AutoModelForSequenceClassification.from_pretrained(OUT_ARBERT)
verify_pipe = hf_pipeline(
    'text-classification',
    model = verify_model,
    tokenizer = verify_tok,
    device = 0 if DEVICE.type == 'cuda' else -1,
    top_k = None,
)

FINAL_CHECKS = [
    ('تحذير عاجل الماء المعدني يسبب مرضا خطيرا واطباء يخفون الحقيقه عن المواطنين منذ سنوات',
     'not_credible', 'conspiracy health claim — must be FAKE'),
    ('اعلنت وزاره التعليم اتفاقيه تعاون مع عدد من الجامعات الدوليه لتطوير المناهج الدراسيه',
     'credible', 'institutional announcement — must be REAL'),
    ('ثعبان عملاق يبتلع مدينه كامله في ظاهره لم يسبق لها مثيل في تاريخ البشريه',
     'not_credible', 'absurd claim — must be FAKE'),
    ('اعلنت الحكومه عن خطه اقتصاديه جديده لدعم الشركات الصغيره والمتوسطه في القطاعات الحيويه',
     'credible', 'government policy news — must be REAL'),
]

print('\n── Final disk-reload sanity checks ──')
all_ok = True
for text, expected, description in FINAL_CHECKS:
    scores = verify_pipe(text[:512])[0]
    best = max(scores, key=lambda x: x['score'])
    pred_lbl = best['label']
    ok = pred_lbl == expected
    all_ok = all_ok and ok
    status = "OK" if ok else "WRONG"
    print(f'  [{status}] {description}')
    print(f'      expected={expected:12s} got={pred_lbl:12s} ({best["score"]:.1%})')

if all_ok:
    print('\nAll final checks passed. Model is correct and ready to download.')
else:
    raise AssertionError('Disk-reload check FAILED — do not use this model.')


## Cell 17 — Summary

In [ ]:
print("Arabic Fake News Detection - AFND / ARBERT Notebook")
print("=" * 60)
print("Model      : ARBERT fine-tuned on AFND")
print("Best for   : formal Modern Standard Arabic (MSA) news")
print("Saved to   : /kaggle/working/model_arbert/")
print("Benchmark  : 98.8% accuracy (Nassif et al. 2022)")
print("-" * 60)
print("Data       : AFND (~607k articles, binary after dropping 'undecided')")
print("Labels     : credible=0, not_credible=1")
print("Split      : 80% train / 10% val / 10% test (stratified)")
print("-" * 60)
print("Next steps:")
print("  1. Set HF_USERNAME and uncomment push_model() call")
print("  2. Pair with the MARBERTv2 notebook for social-media coverage")
print("  3. Wrap the detector in a FastAPI or Gradio endpoint")
print("  4. Quantise with ONNX for faster CPU inference")
